# RACE Reading Comprehension EDA

This notebook checks split quality, label balance, text lengths, frequent words, binary expansion balance, and simple cosine-similarity feature behavior.

## 1. Setup and Data Loading

Load the train, validation, and test CSVs, then inspect shape, dtypes, and null counts to confirm the files match the expected schema.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"

split_paths = {
    "train": DATA_DIR / "train.csv",
    "val": DATA_DIR / "val.csv",
    "test": DATA_DIR / "test.csv",
}

splits = {name: pd.read_csv(path) for name, path in split_paths.items()}
for name, df in splits.items():
    print(f"\n{name.upper()} split")
    print("Shape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)
    print("\nNull counts:")
    print(df.isna().sum())

all_df = pd.concat(
    [df.assign(split=name) for name, df in splits.items()],
    ignore_index=True,
)

expected_cols = ["id", "article", "question", "A", "B", "C", "D", "answer"]
missing_by_split = {name: sorted(set(expected_cols) - set(df.columns)) for name, df in splits.items()}
print("\nMissing expected columns by split:")
print(missing_by_split)

## 2. Missing Values Report

Report missing values as both counts and percentages so data cleaning needs are visible before modeling.

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": all_df[expected_cols].isna().sum(),
    "missing_pct": all_df[expected_cols].isna().mean() * 100,
}).sort_values("missing_count", ascending=False)

missing_report["missing_pct"] = missing_report["missing_pct"].round(3)
missing_report

## 3. Answer Distribution

Check whether A/B/C/D answers are close to balanced. For RACE-style multiple choice, a healthy split should be near 25% for each class.

In [ ]:
answer_pct = all_df["answer"].value_counts(normalize=True).reindex(["A", "B", "C", "D"]) * 100
answer_pct = answer_pct.fillna(0)

plt.figure(figsize=(7, 4))
ax = sns.barplot(x=answer_pct.index, y=answer_pct.values, color="steelblue")
ax.axhline(25, color="black", linestyle="--", linewidth=1, label="25% target")
ax.set_title("Answer Distribution")
ax.set_xlabel("Answer")
ax.set_ylabel("Percentage")
ax.legend()
for i, value in enumerate(answer_pct.values):
    ax.text(i, value + 0.4, f"{value:.1f}%", ha="center")
plt.ylim(0, max(30, answer_pct.max() + 4))
plt.show()

print(answer_pct.round(2))
print(f"Max deviation from 25%: {(answer_pct - 25).abs().max():.2f} percentage points")

## 4. Article Length Distribution

Measure article word counts to understand passage length and detect very long examples that may affect vectorization and runtime.

In [ ]:
def word_count(text):
    return len(str(text).split())

all_df["article_word_count"] = all_df["article"].map(word_count)

plt.figure(figsize=(8, 4))
sns.histplot(all_df["article_word_count"], bins=40, color="seagreen")
plt.title("Article Length Distribution")
plt.xlabel("Article word count")
plt.ylabel("Rows")
plt.show()

all_df["article_word_count"].describe().round(2)

## 5. Question Length Distribution

Inspect question length because unusually short or long questions can affect matching between question, article, and options.

In [ ]:
all_df["question_word_count"] = all_df["question"].map(word_count)

plt.figure(figsize=(8, 4))
sns.histplot(all_df["question_word_count"], bins=30, color="darkorange")
plt.title("Question Length Distribution")
plt.xlabel("Question word count")
plt.ylabel("Rows")
plt.show()

all_df["question_word_count"].describe().round(2)

## 6. Option Length Distribution

Compare option lengths across A/B/C/D to check whether one option position is systematically longer or shorter.

In [ ]:
option_lengths = []
for label in ["A", "B", "C", "D"]:
    option_lengths.append(pd.DataFrame({
        "option": label,
        "word_count": all_df[label].map(word_count),
    }))

option_lengths_df = pd.concat(option_lengths, ignore_index=True)

plt.figure(figsize=(8, 4))
for label in ["A", "B", "C", "D"]:
    subset = option_lengths_df.loc[option_lengths_df["option"] == label, "word_count"]
    sns.kdeplot(subset, label=label, linewidth=2)
plt.title("Option Text Length Distribution")
plt.xlabel("Option word count")
plt.ylabel("Density")
plt.legend(title="Option")
plt.show()

option_lengths_df.groupby("option")["word_count"].describe().round(2)

## 7. Top Article Words

Remove common stopwords and count frequent article terms to see dominant vocabulary in the passages.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "he", "in", "is", "it", "its", "of", "on", "that", "the",
    "to", "was", "were", "will", "with", "this", "these", "those", "or",
    "but", "not", "they", "their", "them", "his", "her", "she", "you",
    "your", "we", "our", "i", "me", "my", "have", "had", "can", "could",
    "would", "should", "there", "then", "than", "so", "if", "about", "into",
    "out", "up", "down", "over", "after", "before", "more", "most", "some",
    "such", "also", "one", "two", "new", "when", "where", "who", "what",
    "which", "why", "how"
}

unique_articles = all_df[["id", "article"]].drop_duplicates()
tokens = []
for text in unique_articles["article"]:
    words = re.findall(r"[a-z]+", str(text).lower())
    tokens.extend([word for word in words if word not in STOPWORDS and len(word) > 2])

top_words = pd.DataFrame(Counter(tokens).most_common(20), columns=["word", "count"])

plt.figure(figsize=(8, 6))
sns.barplot(data=top_words, y="word", x="count", color="slateblue")
plt.title("Top 20 Article Words")
plt.xlabel("Count")
plt.ylabel("Word")
plt.show()

top_words

## 8. Binary Expansion Class Balance

Each multiple-choice row expands into four binary examples: one correct option and three incorrect options. This should produce about 25% positive and 75% negative labels.

In [ ]:
binary_rows = []
for _, row in all_df.iterrows():
    for option in ["A", "B", "C", "D"]:
        binary_rows.append({
            "option": option,
            "label": int(row["answer"] == option),
        })

binary_df = pd.DataFrame(binary_rows)
binary_stats = pd.DataFrame({
    "count": binary_df["label"].value_counts().sort_index(),
    "percentage": binary_df["label"].value_counts(normalize=True).sort_index() * 100,
})
binary_stats.index = ["label=0", "label=1"]
binary_stats["percentage"] = binary_stats["percentage"].round(2)
binary_stats

## 9. Cosine Feature Correlation

Compute three simple bag-of-words cosine features on a 500-row sample and inspect whether they are strongly correlated.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def text_counter(text):
    return Counter(tokenize(text))

def counter_cosine(left, right):
    if not left or not right:
        return 0.0
    shared = set(left) & set(right)
    dot = sum(left[token] * right[token] for token in shared)
    left_norm = math.sqrt(sum(value * value for value in left.values()))
    right_norm = math.sqrt(sum(value * value for value in right.values()))
    return dot / (left_norm * right_norm) if left_norm and right_norm else 0.0

sample_df = all_df.sample(n=min(500, len(all_df)), random_state=RANDOM_STATE)
feature_rows = []

for _, row in sample_df.iterrows():
    article_vec = text_counter(row["article"])
    question_vec = text_counter(row["question"])
    for option in ["A", "B", "C", "D"]:
        option_vec = text_counter(row[option])
        feature_rows.append({
            "article_question": counter_cosine(article_vec, question_vec),
            "article_option": counter_cosine(article_vec, option_vec),
            "question_option": counter_cosine(question_vec, option_vec),
        })

cosine_features = pd.DataFrame(feature_rows)
corr = cosine_features.corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="vlag", center=0, vmin=-1, vmax=1, square=True)
plt.title("Cosine Feature Correlation")
plt.show()

cosine_features.describe().round(3)

## 10. Article Length Outliers

Flag rows where article length is more than three standard deviations above the mean, since very long passages can skew feature extraction.

In [ ]:
mean_len = all_df["article_word_count"].mean()
std_len = all_df["article_word_count"].std()
threshold = mean_len + 3 * std_len

outliers = all_df.loc[
    all_df["article_word_count"] > threshold,
    ["split", "id", "article_word_count", "question", "answer"],
].sort_values("article_word_count", ascending=False)

print(f"Mean article length: {mean_len:.2f}")
print(f"Std article length : {std_len:.2f}")
print(f"Outlier threshold  : {threshold:.2f} words")
print(f"Outlier rows       : {len(outliers)}")

outliers.head(25)